# Visa Assistant Chatbot (RAG)
This notebook builds a PDF-based RAG pipeline using LangChain, HuggingFace embeddings, ChromaDB, and OpenAI/Groq.

## 1. Verify Python Environment

In [ ]:
import sys
print(sys.executable)

## 2. Load PDFs

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("pdfs_visa")
documents = loader.load()

print("Total pages:", len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:1000])

## 3. Split Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

## 4. HuggingFace Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## 5. Add Metadata

In [ ]:
import os

for chunk in chunks:
    source = chunk.metadata["source"]
    chunk.metadata["country"] = os.path.basename(os.path.dirname(source))
    chunk.metadata["document"] = os.path.basename(source)

print(chunks[0].metadata)

## 6. Create Chroma Vector Database

In [ ]:
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="visa_db"
)

print(vector_db._collection.count())

## 7. Retriever

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k":3})

query = "What are the financial requirements for Australia?"
results = retriever.invoke(query)

for doc in results:
    print(doc.metadata)
    print(doc.page_content)
    print("="*80)

## 8. LLM Provider Helper

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
# os.environ["GROQ_API_KEY"] = "YOUR_KEY"

def get_llm(provider="groq"):
    provider = provider.lower()

    if provider == "groq":
        return ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0
        )

    elif provider == "openai":
        return ChatOpenAI(
            model="gpt-4.1-mini",
            temperature=0
        )

    raise ValueError("Provider must be 'groq' or 'openai'")

llm = get_llm("groq")

## 9. Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI Visa Assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

Answer:
""")

## 10. Generate Response

In [ ]:
context = "\n\n".join(doc.page_content for doc in results)

messages = prompt.invoke({
    "context": context,
    "question": query
})

response = llm.invoke(messages)

print(response.content)

## Next Steps
- Add WebBaseLoader
- Merge PDFs + Website data
- Metadata filtering
- Streamlit UI
- Deploy